# 决策树第九课：决策树剪枝

本课目标：理解为什么决策树需要剪枝，以及预剪枝、后剪枝分别在做什么。

一句话版：**剪枝就是有意识地限制树的复杂度，防止它把训练集记得太死。**

## 1. 为什么需要剪枝？

决策树如果不加限制，会不断分裂节点，直到每个叶子节点都非常纯。

这在训练集上看起来很好，但可能出现过拟合：

$$\text{训练集准确率很高，验证集或测试集效果变差。}$$

我们在 Titanic 案例里已经看到过类似现象：

| max_depth | 训练集准确率 | 验证集准确率 |
| ---: | ---: | ---: |
| 3 | 0.8287 | 0.8268 |
| 4 | 0.8357 | 0.8324 |
| None | 0.9817 | 0.7710 |

当 `max_depth=None` 时，树几乎可以无限生长，训练集准确率非常高，但验证集准确率明显下降。

这就是剪枝要解决的问题。

## 2. 什么是剪枝？

剪枝可以理解成：把过于复杂、对泛化没有帮助的分支去掉。

一棵很深的树可能长这样：

```text
Sex?
├── male -> Pclass?
│          ├── 1 -> Age?
│          │       ├── ...
│          │       └── ...
│          └── 3 -> Fare?
│                  ├── ...
│                  └── ...
└── female -> ...
```

剪枝后可能变成：

```text
Sex?
├── male -> Pclass?
└── female -> Pclass?
```

它牺牲了一点训练集拟合能力，但换来更好的泛化能力。

## 3. 剪枝分为两类

决策树剪枝主要分为：

1. 预剪枝：树还没长太深时，就提前阻止它继续分裂。
2. 后剪枝：先让树充分生长，然后再把不必要的分支剪掉。

可以这样记：

$$\text{预剪枝：先拦住。}$$

$$\text{后剪枝：长完再修。}$$

## 4. 预剪枝 Pre-Pruning

预剪枝是在建树过程中提前设置限制条件，让树不要长得太复杂。

常见预剪枝参数：

| 参数 | 含义 |
| --- | --- |
| `max_depth` | 限制树的最大深度 |
| `min_samples_split` | 一个节点至少有多少样本才允许继续分裂 |
| `min_samples_leaf` | 一个叶子节点至少要有多少样本 |
| `max_leaf_nodes` | 限制最多叶子节点数量 |
| `min_impurity_decrease` | 分裂带来的不纯度下降必须足够大 |

例如：

```python
DecisionTreeClassifier(max_depth=4, min_samples_leaf=5)
```

意思是：

- 树最多长 4 层。
- 每个叶子节点至少要有 5 个样本。

这就是预剪枝。

## 5. 预剪枝的优缺点

优点：

- 简单直接。
- 训练速度更快。
- 可以防止树长得过深。
- sklearn 中很常用。

缺点：

- 可能过早停止分裂。
- 有些后面本来能带来好结果的分支，可能还没长出来就被拦住了。

这叫欠拟合风险：

$$\text{树太浅，复杂规律学不到。}$$

## 6. 后剪枝 Post-Pruning

后剪枝的思路是：先让树尽量长出来，然后再检查哪些分支其实没有必要。

流程大致是：

1. 先生成一棵较完整的树。
2. 从底部开始尝试剪掉某些子树。
3. 如果剪掉后验证集效果没有变差，甚至变好，就保留剪枝。
4. 反复进行，直到不能继续剪。

直觉上：

$$\text{如果一个复杂分支对验证集没帮助，就把它换成一个叶子节点。}$$

## 7. CART 的代价复杂度剪枝

CART 常用一种后剪枝方法，叫代价复杂度剪枝 Cost Complexity Pruning。

它的核心目标函数是：

$$R_\alpha(T)=R(T)+\alpha|T|$$

其中：

- $T$：当前这棵树。
- $R(T)$：树在训练数据上的误差。
- $|T|$：叶子节点数量，代表树的复杂度。
- $\alpha$：复杂度惩罚系数。

这个公式的意思是：

$$\text{模型代价 = 训练误差 + 复杂度惩罚}$$

如果 $\alpha$ 越大，模型越讨厌复杂树，剪得就越狠。

在 sklearn 里，对应参数是：

```python
ccp_alpha
```

例如：

```python
DecisionTreeClassifier(ccp_alpha=0.01)
```

## 8. ccp_alpha 怎么影响树？

`ccp_alpha` 控制后剪枝强度。

| ccp_alpha | 树的变化 |
| ---: | --- |
| 0 | 不进行代价复杂度剪枝，树可能更复杂 |
| 较小 | 轻微剪枝 |
| 较大 | 强剪枝，树变浅 |

可以这样理解：

$$\alpha\text{ 越大，复杂度惩罚越强，树越简单。}$$

如果 $\alpha$ 太大，树会被剪得太狠，可能欠拟合。

如果 $\alpha$ 太小，树依然可能过拟合。

## 9. 预剪枝和后剪枝对比

| 对比点 | 预剪枝 | 后剪枝 |
| --- | --- | --- |
| 发生时间 | 建树过程中 | 树长完之后 |
| 思路 | 提前限制树生长 | 先长大，再剪掉不必要分支 |
| 常见参数 | `max_depth`, `min_samples_leaf` | `ccp_alpha` |
| 优点 | 简单、速度快 | 更有机会保留有用结构 |
| 缺点 | 可能提前拦住好分支 | 计算成本更高 |

实际使用时，经常两者结合：

```python
DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=3,
    ccp_alpha=0.001,
)
```

## 10. Titanic 案例里的剪枝理解

我们之前做过不同 `max_depth` 的对比：

| max_depth | 训练集准确率 | 验证集准确率 |
| ---: | ---: | ---: |
| 3 | 0.8287 | 0.8268 |
| 4 | 0.8357 | 0.8324 |
| None | 0.9817 | 0.7710 |

这里的 `max_depth=3`、`max_depth=4` 就是预剪枝。

它们限制树不要长得太深，所以泛化效果更稳。

`max_depth=None` 表示不限制深度，树长得很复杂，训练集准确率很高，但验证集下降。

这说明：

$$\text{树不是越深越好，复杂度要适中。}$$

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 使用 sklearn 自带的鸢尾花数据集做一个轻量示例。
iris = load_iris()
X = iris.data
y = iris.target

# 划分训练集和验证集。
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=22,
    stratify=y,
)

# 不限制深度的树。
deep_tree = DecisionTreeClassifier(random_state=22)
deep_tree.fit(X_train, y_train)

# 预剪枝：限制最大深度。
pre_pruned_tree = DecisionTreeClassifier(max_depth=2, random_state=22)
pre_pruned_tree.fit(X_train, y_train)

# 后剪枝：使用 ccp_alpha。
post_pruned_tree = DecisionTreeClassifier(ccp_alpha=0.02, random_state=22)
post_pruned_tree.fit(X_train, y_train)

models = {
    '不剪枝': deep_tree,
    '预剪枝 max_depth=2': pre_pruned_tree,
    '后剪枝 ccp_alpha=0.02': post_pruned_tree,
}

for name, model in models.items():
    train_acc = accuracy_score(y_train, model.predict(X_train))
    valid_acc = accuracy_score(y_valid, model.predict(X_valid))
    print(name)
    print('  树深度：', model.get_depth())
    print('  叶子节点数：', model.get_n_leaves())
    print('  训练集准确率：', round(train_acc, 4))
    print('  验证集准确率：', round(valid_acc, 4))


## 11. 本课小结

- 决策树太深容易过拟合。
- 剪枝的目标是降低树的复杂度，提高泛化能力。
- 预剪枝是在建树过程中提前限制树的生长。
- 后剪枝是先让树长出来，再剪掉不必要的分支。
- sklearn 中常用 `max_depth`、`min_samples_leaf` 做预剪枝。
- sklearn 中可以用 `ccp_alpha` 做 CART 的代价复杂度剪枝。

记忆方式：

$$\text{预剪枝：别长太疯。后剪枝：长完再修。}$$